In [30]:
import chromadb
import os
import pdfplumber
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [31]:
folder_path = "Documents"

file_list = []

for root, dirs, files in os.walk(folder_path):
    for file in files:
        rel_path = os.path.relpath(os.path.join(root, file), folder_path)
        file_list.append(rel_path)

In [32]:
vector_store = Chroma(
    collection_name = "my_company_policies",
    embedding_function = OpenAIEmbeddings(
        api_key = os.getenv("OPENAI_API_KEY"),
        model = "text-embedding-3-small",
        max_retries = 3,
        request_timeout = 10
    ),
    persist_directory = "./VectorStore"
)

In [33]:
def Load_pdf_to_vectorstore(file_path, chunk_size, chunk_overlap):
    text = ""
    with pdfplumber.open(file_path) as pdf:
        for page in pdf.pages:
            text += page.extract_text() + "\n"


    text_splitter = RecursiveCharacterTextSplitter(chunk_size = chunk_size, chunk_overlap = 0)
    texts = text_splitter.split_text(text)

    doc_id = os.path.basename(file_path)

    documents = [
        Document(
            page_content = chunk,
            metadata = {"source": file_path},
            id = f"{doc_id}_{i}"
        )
        for i, chunk in enumerate(texts)

    ]

    vector_store.add_documents(documents = documents )

In [34]:
# def extract_content_from_pdf(file_path: str) -> str:
#     loader = UnstructuredLoader(file_path, mode = "elements")
#     elements = loader.load()

#     full_text = ""
#     for ele in elements:
#         full_text += ele.page_content.strip() + "\n\n"

#     return full_text

In [35]:
# def split_text(text: str, chunk_size: int, chunk_overlap : int):
#     splitter = RecursiveCharacterTextSplitter(
#         chunk_size = chunk_size,
#         chunk_overlap = chunk_overlap,
#         separators = ["\n\n", "\n", " ", ""]
#     )

#     return splitter.split_text(text)

In [36]:
# def upload_text_chunks_to_vectorstore(chunks, source):
#     documents = [
#         Document(
#             page_content = chunk,
#             metadata = {"source": source},
#         )
#         for chunk in chunks
#     ]
#     vector_store.add_documents(documents)

In [37]:
# def Load_pdf_to_vectorstore(file_path, chunk_size, chunk_overlap):
#     text = extract_content_from_pdf(file_path)
#     chunks = split_text(text, chunk_size, chunk_overlap)
#     upload_text_chunks_to_vectorstore(chunks, file_path)

In [38]:
# def Load_pdf_to_vectorstore(file_path, chunk_size, chunk_overlap):
#     loader = UnstructuredLoader(file_path, mode = "elements")
#     raw_docs = loader.load()
#     raw_text = "\n\n".join(el.text for el in raw_docs)

#     raw_content = [x.page_content for x in raw_text]
#     docs = clean_unstructured_docs(raw_content)
#     text_splitter = RecursiveCharacterTextSplitter(chunk_size = chunk_size, chunk_overlap = chunk_overlap)
#     texts = text_splitter.split_documents(docs)

#     doc_id = os.path.basename(file_path)

#     documents = [
#         Document(
#             page_content = chunk,
#             metadata = {"source": file_path},
#             id = f"{doc_id}_{i}"
#         )
#         for i, chunk in enumerate(texts)

#     ]

#     vector_store.add_documents(documents = documents )

    

In [39]:
chunk_size = 1000
chunk_overlap = 100
for file in file_list:
    full_path = "Documents/" + file
    #print(f"Processing {file}...")
    Load_pdf_to_vectorstore(full_path, chunk_size, chunk_overlap)
    print(f"Uploaded {file} to vector store.")

Uploaded anti-corruption-policy-english.pdf to vector store.
Uploaded cf-compliance-policies-and-procedures.pdf to vector store.
Uploaded code-of-ethics.pdf to vector store.
Uploaded cognizant-global-supplier-diversification-policy.pdf to vector store.
Uploaded cognizant-privacy-notice.pdf to vector store.
Uploaded cognizant-safe-workforce-solution-overview.pdf to vector store.
Uploaded cognizant-vendor-anti-corruption-compliance.pdf to vector store.
Uploaded corporate-governance-guidelines.pdf to vector store.
Uploaded environmental-health-and-safety-policy.pdf to vector store.
Uploaded equal-employment-opportunity-policy.pdf to vector store.
Uploaded human-rights-policy.pdf to vector store.
Uploaded making-ai-responsible-and-effective-codex3974.pdf to vector store.
Uploaded political-activity-policy.pdf to vector store.
Uploaded supplier-standards-of-conduct.pdf to vector store.
Uploaded [OUTDATED] Anti-Corruption Policy v1.2.pdf to vector store.


In [ ]:
# result = vector_store.similarity_search(
#     "What are the four core principles of the Cognizant Code of Ethics?",
#     k=3
# )

In [ ]:
# result

[Document(id='code-of-ethics.pdf_3', metadata={'source': 'Documents/code-of-ethics.pdf'}, page_content='Cognizant code of ethics | 3\nThe four principles: The right way at Cognizant\nHow we do our work defines us.\nWe engineer modern businesses to improve everyday life.\n1. We earn trust.\nThis is our purpose, and how we do it matters more now\nWe continually strive to be a trusted\nbusiness partner and corporate citizen. than ever.\nIn pursuing this goal, we must consistently\nAt the heart of our purpose is a desire to improve not just our clients’ businesses, but how we operate as\nincorporate ethical standards into our day-\na leader in our industry. One way we distinguish ourselves in the market is by maintaining the highest\nto-day business activities\nstandards of integrity. Our reputation and our success depend on it— but it’s more than that. Whether\n2. We do the right thing, the right way. at work, in a development center, on-site with our clients, in our corporate office, or 

In [ ]:
# for ans in result:
#     print(ans.page_content)
#     print("--"*100)

Cognizant code of ethics | 3
The four principles: The right way at Cognizant
How we do our work defines us.
We engineer modern businesses to improve everyday life.
1. We earn trust.
This is our purpose, and how we do it matters more now
We continually strive to be a trusted
business partner and corporate citizen. than ever.
In pursuing this goal, we must consistently
At the heart of our purpose is a desire to improve not just our clients’ businesses, but how we operate as
incorporate ethical standards into our day-
a leader in our industry. One way we distinguish ourselves in the market is by maintaining the highest
to-day business activities
standards of integrity. Our reputation and our success depend on it— but it’s more than that. Whether
2. We do the right thing, the right way. at work, in a development center, on-site with our clients, in our corporate office, or in our everyday lives,
Our clients, shareholders, and communities integrity is core to who we are.
-------------------

In [43]:
# vector_store.delete_collection()